***

Preparing Workspace

***

In [ ]:
# General
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import math
import seaborn as sns

# Plotting
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.offline import plot
import plotly.subplots as sp
from plotly.subplots import make_subplots
pd.options.display.float_format = '{:.0f}'.format


## See if you can bold outline of boxes in charts for sac/yuba compared to other Peer MSA's

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

# Set file paths
if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    
if user in ['jchoy', 'aazawii']:
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Python Projects', 'Regional-Monitoring', 'Indicator_Gen')

path_config0 = os.path.join(path_git, 'config')
path_config  = os.path.join(path_git, 'Data', 'BLS', 'config')


print(user)
print(path_git)

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0,     'Functions.py')).read())
exec(open(os.path.join(path_config , 'BLS Functions.py')).read())

In [ ]:
# Set parameters for export file
path_plots = r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring"
print('Export Location: ' + path_plots)

***

Jobs_1

***

In [ ]:
# Set indicator
indicator_name = 'Jobs_1'

file_name1 = f"{indicator_name} MSA BLS SM.xlsx"
file_name2 = f"{indicator_name} National BLS CE.xlsx"

# Impoting data
df_msa = pd.read_excel(os.path.join(path_plots, 'Data', file_name1), sheet_name = 'MSA')
df_nat = pd.read_excel(os.path.join(path_plots, 'Data', file_name2), sheet_name = 'National')

df_msa = df_msa.rename(columns = {'MSA':'Geography'})
df_nat = df_nat.rename(columns = {'MSA':'Geography'})

df_jobs = pd.concat([df_msa, df_nat])
df_jobs = df_jobs.rename(columns = {'Total Jobs':'Value'})
df_jobs = df_jobs.reset_index(drop = True)


display(df_msa.head(), df_nat.head(), df_jobs.head())

Monthly Job Growth

In [ ]:


## Importing ---
indicator_name = 'Jobs_1'

file_name1 = f"{indicator_name} MSA BLS SM.xlsx"
file_name2 = f"{indicator_name} National BLS CE.xlsx"

df_msa = pd.read_excel(os.path.join(path_plots, 'Data', file_name1), sheet_name = 'MSA')
df_nat = pd.read_excel(os.path.join(path_plots, 'Data', file_name2), sheet_name = 'National')

df_msa = df_msa.rename(columns = {'MSA':'Geography'})
df_nat = df_nat.rename(columns = {'MSA':'Geography'})

df_jobs = pd.concat([df_msa, df_nat])
df_jobs = df_jobs.rename(columns = {'Total Jobs':'Value'})
df_jobs = df_jobs.reset_index(drop = True)



## Organizing ---

df_plot = df_jobs.copy()

# Growth rate
df_plot = df_plot[df_plot['Sector'] == 'All']
df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, True])
df_plot['Jobs_GR'] = df_plot['Value'].pct_change()*100
df_plot.loc[df_plot['date_'] == '2000-01-01', 'Jobs_GR'] = np.nan
df_plot.loc[df_plot['Jobs_GR'] == np.inf, 'Jobs_GR'] = np.nan
df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, False])

# SACOG roll up
conditions = [   
       df_plot['Geography'].str.contains('Sac|Yuba')
    , ~df_plot['Geography'].str.contains('Sac|Yuba|National')
    ,  df_plot['Geography'].str.contains('National')
             ]
choices = ['SACOG', 'Peer MSA', 'National']
df_plot['Groups'] = np.select(conditions, choices)

wm = lambda x: np.average(x, weights = df_plot.loc[x.index, "Value"]) # weighted average
df_plot = df_plot.groupby(['date_', 'Groups'], as_index = False).agg(Value = ('Value', 'sum'), Jobs_GR = ('Jobs_GR', wm))
df_plot = df_plot.sort_values(['Groups', 'date_'], ascending = [True, False])
df_plot = df_plot.rename(columns = {'Jobs_GR':'Growth Rate'})

display(df_plot.head())


## Plotting ---

color_map = {
     "SACOG":"#9DC209",
     "National": "#1F45FC",
     "Peer MSA": "#1E90FF"
}

fig = px.line(df_plot, x='date_', y='Growth Rate', color='Groups', color_discrete_map=color_map)

title = 'Monthly Job Growth: Sacramento and other Mid-Sized Metro Areas'
fig.update_layout(xaxis_title = 'Date')
fig.update_yaxes(tick0=0, dtick=2, ticksuffix='%')
fig.update_xaxes(dtick="M48", tickformat="%b\n%Y", ticklabelmode="period")

fig.update_layout(title=title, legend_title=None, template=template, font_family=font_family)
fig.show()

# fig.write_html(os.path.join(path_plots, 'Jobs_1_Monthly Job Growth_line.html'))

Annual Job Growth (September)

In [ ]:

## Importing ---
indicator_name = 'Jobs_1'

file_name1 = f"{indicator_name} MSA BLS SM.xlsx"
file_name2 = f"{indicator_name} National BLS CE.xlsx"

df_msa = pd.read_excel(os.path.join(path_plots, 'Data', file_name1), sheet_name = 'MSA')
df_nat = pd.read_excel(os.path.join(path_plots, 'Data', file_name2), sheet_name = 'National')

df_msa = df_msa.rename(columns = {'MSA':'Geography'})
df_nat = df_nat.rename(columns = {'MSA':'Geography'})

df_jobs = pd.concat([df_msa, df_nat])
df_jobs = df_jobs.rename(columns = {'Total Jobs':'Value'})
df_jobs = df_jobs.reset_index(drop = True)


## Oragnizing ---

df_plot = df_jobs.copy()
month = '09'

df_plot = df_plot[df_plot['Sector'] == 'All']
df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, True])
df_plot = df_plot[df_plot['date_'].str.contains(f'{month}-01')]
df_plot = df_plot[~df_plot['date_'].str.contains('2000-01-01')]
df_plot = df_plot[~df_plot['date_'].str.contains(f'20{month}-01-01')]
df_plot['Jobs_GR'] = df_plot['Value'].pct_change()*100
df_plot.loc[df_plot['date_'] == f'2000-{month}-01', 'Jobs_GR'] = np.nan
df_plot.loc[df_plot['Jobs_GR'] == np.inf, 'Jobs_GR'] = np.nan
df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, False])
df_plot = df_plot[~df_plot['Jobs_GR'].isna()]
df_plot = df_plot.reset_index(drop = True)

wm = lambda x: np.average(x, weights = df_plot.loc[x.index, "Value"]) # weighted average

conditions = [   
       df_plot['Geography'].str.contains('Sac|Yuba')
    , ~df_plot['Geography'].str.contains('Sac|Yuba|National')
    ,  df_plot['Geography'].str.contains('National')
             ]
choices = ['SACOG', 'Peer MSA', 'National']
df_plot['Groups'] = np.select(conditions, choices)
df_plot = df_plot.groupby(['date_', 'Groups'], as_index = False).agg(Value = ('Value', 'sum'), Jobs_GR = ('Jobs_GR', wm))
df_plot = df_plot.sort_values(['Groups', 'date_'], ascending = [True, False])

conditions = [   
         df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2000, 2007, 1)))
       , df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2008, 2011, 1)))
       , df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2012, 2019, 1)))
       , df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2020, 2020, 1)))
       , df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2021, 2024, 1)))
             ]
choices = ["Pre Recession<br>(2000-2008)", "Recession<br>(2008-2011)", "Post Recession<br>(2011-2020)", "Covid<br>(2020)", "Post Covid<br>(2020-2024)"]
df_plot["Period"] = np.select(conditions, choices)

df_plot = df_plot.dropna()
df_plot1 = df_plot.groupby(['Groups', 'Period'], as_index = False)['Jobs_GR'].agg(np.mean)
df_plot2 = df_plot.groupby(['Groups'          ], as_index = False)['Jobs_GR'].agg(np.mean)
df_plot2['Period'] = 'Total<br>(2000-2024)'
df_plot = pd.concat([df_plot1, df_plot2])
categories = ["Total<br>(2000-2024)", "Pre Recession<br>(2000-2008)", "Recession<br>(2008-2011)", "Post Recession<br>(2011-2020)", "Covid<br>(2020)", "Post Covid<br>(2020-2024)"]
df_plot['Period_Sort'] = pd.Categorical(df_plot['Period'], categories)
df_plot = df_plot.sort_values(by = ['Groups', 'Period_Sort'], ascending = [True, True])
df_plot = df_plot.drop(['Period_Sort'], axis = 1)
df_plot = df_plot.rename(columns = {'Jobs_GR':'Growth Rate', 'Period':'Time Period'})

display(df_plot.head())


## Plotting ---

color_map = {
     "SACOG":"#9DC209",
     "National": "#1F45FC",
     "Peer MSA": "#1E90FF"
}

df_plot['Growth Rate'] = round(df_plot["Growth Rate"], 2)
# df_plot = df_plot.rename(columns = {'Group':'Geography'})
fig1 = px.bar(df_plot, x='Time Period', y='Growth Rate'
             , color='Groups'
             , color_discrete_map=color_map
             , barmode='group'
             , hover_name = 'Time Period')
fig1.update_yaxes(tick0=0, dtick=2, ticksuffix='%')
fig1.update_xaxes(tickangle=0)
fig1.update_layout(xaxis_title=None, xaxis=dict(tickfont = dict(size=11)))

title = 'Annual Job Growth Comparison: Sacramento, National, and other Mid-Sized Metro Areas (September)'
fig1.update_layout(legend_title=None, title=title, template=template, font_family=font_family)
fig1.update_traces(hovertemplate="Growth Rate: %{y}")
fig1.show()

# fig1.write_html(os.path.join(path_plots, 'Jobs_1_bar.html'))

Annual Job Growth (September)

In [ ]:

## Importing ---
indicator_name = 'Jobs_1'

file_name1 = f"{indicator_name} MSA BLS SM.xlsx"
file_name2 = f"{indicator_name} National BLS CE.xlsx"

df_msa = pd.read_excel(os.path.join(path_plots, 'Data', file_name1), sheet_name = 'MSA')
df_nat = pd.read_excel(os.path.join(path_plots, 'Data', file_name2), sheet_name = 'National')

df_msa = df_msa.rename(columns = {'MSA':'Geography'})
df_nat = df_nat.rename(columns = {'MSA':'Geography'})

df_jobs = pd.concat([df_msa, df_nat])
df_jobs = df_jobs.rename(columns = {'Total Jobs':'Value'})
df_jobs = df_jobs.reset_index(drop = True)


## Organizing ---

df_plot = df_jobs.copy()
month = '09'

df_plot = df_plot[df_plot['Sector'] == 'All']
df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, True])
df_plot = df_plot[df_plot['date_'].str.contains(f'{month}-01')]
df_plot = df_plot[~df_plot['date_'].str.contains('2000-01-01')]
df_plot = df_plot[~df_plot['date_'].str.contains(f'20{month}-01-01')]
df_plot['Jobs_GR'] = df_plot['Value'].pct_change()*100
df_plot.loc[df_plot['date_'] == f'2000-{month}-01', 'Jobs_GR'] = np.nan
df_plot.loc[df_plot['Jobs_GR'] == np.inf, 'Jobs_GR'] = np.nan
df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, False])
df_plot = df_plot[~df_plot['Jobs_GR'].isna()]
df_plot = df_plot.reset_index(drop = True)

wm = lambda x: np.average(x, weights = df_plot.loc[x.index, "Value"]) # weighted average

conditions = [   
       df_plot['Geography'].str.contains('Sac|Yuba')
    , ~df_plot['Geography'].str.contains('Sac|Yuba|National')
    ,  df_plot['Geography'].str.contains('National')
             ]
choices = ['SACOG', 'Peer MSA', 'National']
df_plot['Groups'] = np.select(conditions, choices)
df_plot = df_plot.groupby(['date_', 'Groups'], as_index = False).agg(Value = ('Value', 'sum'), Jobs_GR = ('Jobs_GR', wm))
df_plot = df_plot.sort_values(['Groups', 'date_'], ascending = [True, False])


df_plot = df_plot.dropna()
df_plot = df_plot.sort_values(by = ['Groups', 'date_'], ascending = [True, True])
df_plot = df_plot.rename(columns = {'Jobs_GR':'Growth Rate'})

df_plot['Growth Rate'] = round(df_plot["Growth Rate"], 2)
df_plot = df_plot.rename(columns = {'Groups':'Geography'})

display(df_plot.head())


## Plotting ---

color_map = {
    'Sacramento-Roseville-Folsom, CA': "#9DC209"
    , 'Yuba City, CA': "#9DC209"
    , 'National': "#1F45FC"
    , 'Austin-Round Rock-Georgetown, TX': "#1E90FF"
    , 'Charlotte-Concord-Gastonia, NC-SC': "#1E90FF"
    , 'Cincinnati, OH-KY-IN': "#1E90FF"
    , 'Cleveland-Elyria, OH': "#1E90FF"
    , 'Columbus, OH': "#1E90FF"
    , 'Detroit-Warren-Dearborn, MI': "#1E90FF"
    , 'Indianapolis-Carmel-Anderson, IN': "#1E90FF"
    , 'Kansas City, MO-KS': "#1E90FF"
    , 'Miami-Fort Lauderdale-Pompano Beach, FL': "#1E90FF"
    , 'Orlando-Kissimmee-Sanford, FL': "#1E90FF"
    , 'Phoenix-Mesa-Chandler, AZ': "#1E90FF"
    , 'Pittsburgh, PA': "#1E90FF"
    , 'Portland-Vancouver-Hillsboro, OR-WA': "#1E90FF"
    , 'Riverside-San Bernardino-Ontario, CA': "#1E90FF"
    , 'Salt Lake City, UT': "#1E90FF"
    , 'San Antonio-New Braunfels, TX': "#1E90FF"
    , 'San Diego-Chula Vista-Carlsbad, CA': "#1E90FF"
    , 'San Francisco-Oakland-Berkeley, CA': "#1E90FF"
    , 'San Jose-Sunnyvale-Santa Clara, CA': "#1E90FF"
    , 'St. Louis, MO-IL': "#1E90FF"
    , 'Tampa-St. Petersburg-Clearwater, FL': "#1E90FF"
}

fig2 = px.line(df_plot, x = 'date_', y = 'Growth Rate', color = 'Geography', color_discrete_map=color_map)
fig2.update_yaxes(tick0=0, dtick=1)
fig2.add_vline(x = '2009-01-01', line_dash = 'dash')
fig2.add_vline(x = '2012-01-01', line_dash = 'dash')
fig2.add_vline(x = '2020-01-01', line_dash = 'dash')
fig2.add_vline(x = '2021-03-01', line_dash = 'dash')
fig2.add_annotation(x='2005-01-01', y = -6.5, text="Pre-Recession" , showarrow= False)
fig2.add_annotation(x='2010-07-01', y = -6.5, text="Recession"     , showarrow= False)
fig2.add_annotation(x='2016-01-01', y = -6.5, text="Post-Recession", showarrow= False)
fig2.add_annotation(x='2020-07-20', y = -6.5, text="Covid"         , showarrow= False)
fig2.add_annotation(x='2022-09-01', y = -6.5, text="Post-Covid"    , showarrow= False)
fig2.add_hline(y = 0, line_dash = 'dash', line_color = 'gray')
fig2.update_xaxes(dtick="M48", tickformat="%Y", ticklabelmode="period")
fig2.update_layout(xaxis_title  = 'Year')


title = 'Monthly Job Growth: Sacramento and other Mid-Sized Metro Areas (September)'
fig2.update_layout(legend_title=None, title=title, template=template, font_family=font_family)
fig2.update_traces(hovertemplate="Year: %{x}<br>Growth Rate: %{y}")


fig2.show()

In [ ]:
# https://stackoverflow.com/questions/56727843/how-can-i-create-subplots-with-plotly-express

fig1_traces = []
fig2_traces = []


for trace in range(len(fig1["data"])):
    fig1_traces.append(fig1["data"][trace])

for trace in range(len(fig2["data"])):
    fig2["data"][trace]['showlegend'] = False
    fig2_traces.append(fig2["data"][trace])
    



# Create a 1x2 subplot
fig = sp.make_subplots(rows = 1, cols = 2
                       , subplot_titles=('<span style="font-size: 13px;">By Time Period (September to September)</span>', 
                                         '<span style="font-size: 13px;">By Year (September to September)</span>')
                      )

# Get the Express fig broken down as traces and add the traces to the proper plot within the subplot
for traces in fig1_traces:
    fig.append_trace(traces, row = 1, col = 1)
for traces in fig2_traces:
    fig.append_trace(traces, row = 1, col = 2)



# fig.update_layout(legend_title=None, title='Monthly Job Growth: Sacramento and other Mid-Sized Metro Areas (by Presidential Administration) (November)')
fig.add_vline(x = '2009-01-01', line_dash = 'dash', row=1, col=2, line_color = 'gray')
fig.add_vline(x = '2012-01-01', line_dash = 'dash', row=1, col=2, line_color = 'gray')
fig.add_vline(x = '2020-01-01', line_dash = 'dash', row=1, col=2, line_color = 'gray')
fig.add_vline(x = '2021-03-01', line_dash = 'dash', row=1, col=2, line_color = 'gray')
fig.add_annotation(x='2005-01-01', y = -7, text='<span style="font-size: 7px;">Pre-Recession</span>' , showarrow= False, row=1, col=2)
fig.add_annotation(x='2010-07-01', y = -7, text='<span style="font-size: 7px;">Recession</span>'     , showarrow= False, row=1, col=2)
fig.add_annotation(x='2016-01-01', y = -7, text='<span style="font-size: 7px;">Post-Recession</span>', showarrow= False, row=1, col=2)
fig.add_annotation(x='2020-07-20', y = -7, text='<span style="font-size: 7px;">Covid</span>'         , showarrow= False, row=1, col=2)
fig.add_annotation(x='2022-07-01', y = -7, text='<span style="font-size: 7px;">Post-Covid</span>'    , showarrow= False, row=1, col=2)
fig.add_hline(y = 0, line_dash = 'dash', line_color = 'gray', row=1, col=2)
            

title='Job Growth Comparison: Sacramento, National, and other Mid-Sized Metro Areas'
fig.update_yaxes(tick0=0, dtick=2, ticksuffix='%', range = [-7,7])
fig.update_xaxes(tickangle=0)
fig.update_xaxes(showticklabels=False, showgrid=False, row=1, col=2)
fig.update_layout(xaxis_title = None, xaxis = dict(tickfont = dict(size=8)))
fig.update_layout(legend_title=None, title=title, template=template, font_family=font_family)
        
fig.show()


# fig.write_html(os.path.join(path_plots, 'Jobs_1_bar_and_line.html'))


***

Jobs_3

***

In [ ]:
# Set indicator
indicator_name = 'Jobs_3'

# Import data
file_name1 = f"{indicator_name} MSA BLS SM.xlsx"
file_name2 = f"{indicator_name} National BLS CE.xlsx"

df_msa = pd.read_excel(os.path.join(path_plots, 'Data', file_name1), sheet_name = 'Goods and Services')
df_nat = pd.read_excel(os.path.join(path_plots, 'Data', file_name2), sheet_name = 'Goods and Services')

display(df_msa.head(), df_nat.head())

In [ ]:
## Importing ---

indicator_name = 'Jobs_3'

file_name1 = f"{indicator_name} MSA BLS SM.xlsx"
file_name2 = f"{indicator_name} National BLS CE.xlsx"

df_msa = pd.read_excel(os.path.join(path_plots, 'Data', file_name1), sheet_name = 'Goods and Services')
df_nat = pd.read_excel(os.path.join(path_plots, 'Data', file_name2), sheet_name = 'Goods and Services')


## Organizing ---
df_plot = df_msa.copy()

df_plot = df_plot.rename(columns = {'MSA':'Geography'})
df_plot = pd.concat([df_plot, df_nat])

df_plot = df_plot[df_plot['Sector'] != 'All']
df_plot = df_plot[df_msa['date_'] == '2024-04-01']
df_plot['Year'] = pd.to_datetime(df_plot['date_'])
df_plot['Year'] = df_plot['Year'].dt.year
df_plot = df_plot.drop(['date_'], axis = 1)
df_plot = df_plot.set_index('Year').reset_index()
df_plot = df_plot.reset_index(drop = True)

def remove_metro(x):
    x = re.sub(' Metro Area', '', x)
    return x
    
df_plot['Geography'] = df_plot['Geography'].apply(remove_metro)

df_plot['Geography_sort'] = pd.Categorical(df_plot['Geography'], [
    'Sacramento-Roseville-Folsom, CA'
    , 'Yuba City, CA'
    , 'National'
    , 'Austin-Round Rock-Georgetown, TX'
    , 'Charlotte-Concord-Gastonia, NC-SC'
    , 'Cincinnati, OH-KY-IN'
    , 'Cleveland-Elyria, OH'
    , 'Columbus, OH'
    , 'Detroit-Warren-Dearborn, MI'
    , 'Indianapolis-Carmel-Anderson, IN'
    , 'Kansas City, MO-KS'
    , 'Miami-Fort Lauderdale-Pompano Beach, FL'
    , 'Orlando-Kissimmee-Sanford, FL'
    , 'Phoenix-Mesa-Chandler, AZ'
    , 'Pittsburgh, PA'
    , 'Portland-Vancouver-Hillsboro, OR-WA'
    , 'Riverside-San Bernardino-Ontario, CA'
    , 'Salt Lake City, UT'
    , 'San Antonio-New Braunfels, TX'
    , 'San Diego-Chula Vista-Carlsbad, CA'
    , 'San Francisco-Oakland-Berkeley, CA'
    , 'San Jose-Sunnyvale-Santa Clara, CA'
    , 'St. Louis, MO-IL'
    , 'Tampa-St. Petersburg-Clearwater, FL'
])
    
df_plot = df_plot.sort_values(['Geography_sort', 'Sector'], ascending = [True, True])
df_plot = df_plot.drop(['Geography_sort'], axis = 1)

df_plot = df_plot[df_plot['Sector'] == 'Goods Producing']
df_plot['Percentage'] = round(df_plot['Percentage'], 1)

display(df_plot.head())


## Plotting ---


color_map = {
    'Sacramento-Roseville-Folsom, CA': "#9DC209"
    , 'Yuba City, CA': "#9DC209"
    , 'National': "#1F45FC"
    , 'Austin-Round Rock-Georgetown, TX': "#1E90FF"
    , 'Charlotte-Concord-Gastonia, NC-SC': "#1E90FF"
    , 'Cincinnati, OH-KY-IN': "#1E90FF"
    , 'Cleveland-Elyria, OH': "#1E90FF"
    , 'Columbus, OH': "#1E90FF"
    , 'Detroit-Warren-Dearborn, MI': "#1E90FF"
    , 'Indianapolis-Carmel-Anderson, IN': "#1E90FF"
    , 'Kansas City, MO-KS': "#1E90FF"
    , 'Miami-Fort Lauderdale-Pompano Beach, FL': "#1E90FF"
    , 'Orlando-Kissimmee-Sanford, FL': "#1E90FF"
    , 'Phoenix-Mesa-Chandler, AZ': "#1E90FF"
    , 'Pittsburgh, PA': "#1E90FF"
    , 'Portland-Vancouver-Hillsboro, OR-WA': "#1E90FF"
    , 'Riverside-San Bernardino-Ontario, CA': "#1E90FF"
    , 'Salt Lake City, UT': "#1E90FF"
    , 'San Antonio-New Braunfels, TX': "#1E90FF"
    , 'San Diego-Chula Vista-Carlsbad, CA': "#1E90FF"
    , 'San Francisco-Oakland-Berkeley, CA': "#1E90FF"
    , 'San Jose-Sunnyvale-Santa Clara, CA': "#1E90FF"
    , 'St. Louis, MO-IL': "#1E90FF"
    , 'Tampa-St. Petersburg-Clearwater, FL': "#1E90FF"
}


fig = px.bar(df_plot, y='Geography', x='Percentage'
             , color='Geography'
             , color_discrete_map=color_map
             , orientation='h')


title = 'Goods Producing Share of Total Regional Jobs, 2024'
fig.update_layout(legend_title=None, title=title, template=template, font_family=font_family)
fig.update_layout(xaxis_title = None, yaxis_title = None, yaxis=dict(tickfont = dict(size=12)), xaxis=dict(tickfont = dict(size=12)), showlegend = False)
fig.update_xaxes(tick0=0, dtick=5, ticksuffix='%')
fig.update_traces(hovertemplate='Goods Producing: %{x}')


fig.show()


In [ ]:
## Importing ---

indicator_name = 'Jobs_3'

file_name1 = f"{indicator_name} MSA BLS SM.xlsx"
file_name2 = f"{indicator_name} National BLS CE.xlsx"

df_msa = pd.read_excel(os.path.join(path_plots, 'Data', file_name1), sheet_name = 'Goods and Services')
df_nat = pd.read_excel(os.path.join(path_plots, 'Data', file_name2), sheet_name = 'Goods and Services')


## Organizing ---
df_plot = df_msa.copy()

df_plot = df_plot.rename(columns = {'MSA':'Geography'})
df_plot = pd.concat([df_plot, df_nat])

df_plot = df_plot[df_plot['Sector'] != 'All']
df_plot = df_plot[df_msa['date_'] == '2024-04-01']
df_plot['Year'] = pd.to_datetime(df_plot['date_'])
df_plot['Year'] = df_plot['Year'].dt.year
df_plot = df_plot.drop(['date_'], axis = 1)
df_plot = df_plot.set_index('Year').reset_index()
df_plot = df_plot.reset_index(drop = True)
def remove_metro(x):
    x = re.sub(' Metro Area', '', x)
    return x
df_plot['Geography'] = df_plot['Geography'].apply(remove_metro)

df_plot['Sort'] = pd.Categorical(df_plot['Geography'], [
    'Sacramento-Roseville-Folsom, CA'
    , 'Yuba City, CA'
    , 'Austin-Round Rock-Georgetown, TX'
    , 'Charlotte-Concord-Gastonia, NC-SC'
    , 'Cincinnati, OH-KY-IN'
    , 'Cleveland-Elyria, OH'
    , 'Columbus, OH'
    , 'Detroit-Warren-Dearborn, MI'
    , 'Indianapolis-Carmel-Anderson, IN'
    , 'Kansas City, MO-KS'
    , 'Miami-Fort Lauderdale-Pompano Beach, FL'
    , 'Orlando-Kissimmee-Sanford, FL'
    , 'Phoenix-Mesa-Chandler, AZ'
    , 'Pittsburgh, PA'
    , 'Portland-Vancouver-Hillsboro, OR-WA'
    , 'Riverside-San Bernardino-Ontario, CA'
    , 'Salt Lake City, UT'
    , 'San Antonio-New Braunfels, TX'
    , 'San Diego-Chula Vista-Carlsbad, CA'
    , 'San Francisco-Oakland-Berkeley, CA'
    , 'San Jose-Sunnyvale-Santa Clara, CA'
    , 'St. Louis, MO-IL'
    , 'Tampa-St. Petersburg-Clearwater, FL'
    , 'National'
])
    
df_plot = df_plot.sort_values(['Sort', 'Sector'], ascending = [False, True])
df_plot = df_plot.drop(['Sort'], axis = 1)
df_plot['Percentage'] = round(df_plot['Percentage'], 1)

display(df_plot.head())


## Plotting ---

color_map = {
    'Goods Producing': '#1E90FF'
    , 'Service-Providing':'#9DC209'
}

fig = px.bar(df_plot, y='Geography', x='Percentage'
             , color='Sector'
             , color_discrete_map=color_map
             , orientation='h')



ticktext = []
for geography in df_plot['Geography'].unique():
    if geography in ['Sacramento-Roseville-Folsom, CA', 'Yuba City, CA', 'National']:
        ticktext.append(f'<b>{geography}</b>')
    else:
        ticktext.append(geography)


fig.update_layout(yaxis=dict(tickmode='array', tickvals=df_plot['Geography'].unique(), ticktext=ticktext))

title = 'Economic Structure: Share of Goods Production vs Services by Peer Region, 2024'

fig.update_layout(legend_title=None, title=title, template=template, font_family=font_family)
fig.update_layout(xaxis_title = None, yaxis_title = None, yaxis=dict(tickfont = dict(size=12)), xaxis=dict(tickfont = dict(size=12)))
fig.update_xaxes(tick0=0, dtick=10, ticksuffix='%')
fig.update_traces(hovertemplate='%{x}')
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=-0.1,xanchor="right", x=0.55))

fig.show()
# fig.write_html(os.path.join(path_plots, 'Jobs_3_goods_services.html'))

In [ ]:
## Importing ---

indicator_name = 'Jobs_3'

file_name1 = f"{indicator_name} MSA BLS SM.xlsx"
file_name2 = f"{indicator_name} National BLS CE.xlsx"

df_msa = pd.read_excel(os.path.join(path_plots, 'Data', file_name1), sheet_name = 'Government and Private')
df_nat = pd.read_excel(os.path.join(path_plots, 'Data', file_name2), sheet_name = 'Government and Private')



## Organizing ---
df_plot = df_msa.copy()

df_plot = df_plot.rename(columns = {'MSA':'Geography'})
df_plot = pd.concat([df_plot, df_nat])

df_plot = df_plot[df_plot['Sector'] != 'All']
df_plot = df_plot[df_msa['date_'] == '2024-04-01']
df_plot['Year'] = pd.to_datetime(df_plot['date_'])
df_plot['Year'] = df_plot['Year'].dt.year
df_plot = df_plot.drop(['date_'], axis = 1)
df_plot = df_plot.set_index('Year').reset_index()
df_plot = df_plot.reset_index(drop = True)

def remove_metro(x):
    x = re.sub(' Metro Area', '', x)
    return x
df_plot['Geography'] = df_plot['Geography'].apply(remove_metro)

df_plot['Geography_sort'] = pd.Categorical(df_plot['Geography'], [
    'Sacramento-Roseville-Folsom, CA'
    , 'Yuba City, CA'
    , 'Austin-Round Rock-Georgetown, TX'
    , 'Charlotte-Concord-Gastonia, NC-SC'
    , 'Cincinnati, OH-KY-IN'
    , 'Cleveland-Elyria, OH'
    , 'Columbus, OH'
    , 'Detroit-Warren-Dearborn, MI'
    , 'Indianapolis-Carmel-Anderson, IN'
    , 'Kansas City, MO-KS'
    , 'Miami-Fort Lauderdale-Pompano Beach, FL'
    , 'Orlando-Kissimmee-Sanford, FL'
    , 'Phoenix-Mesa-Chandler, AZ'
    , 'Pittsburgh, PA'
    , 'Portland-Vancouver-Hillsboro, OR-WA'
    , 'Riverside-San Bernardino-Ontario, CA'
    , 'Salt Lake City, UT'
    , 'San Antonio-New Braunfels, TX'
    , 'San Diego-Chula Vista-Carlsbad, CA'
    , 'San Francisco-Oakland-Berkeley, CA'
    , 'San Jose-Sunnyvale-Santa Clara, CA'
    , 'St. Louis, MO-IL'
    , 'Tampa-St. Petersburg-Clearwater, FL'
    , 'National'
])
    
df_plot = df_plot.sort_values(['Geography_sort', 'Sector'], ascending = [True, True])
df_plot = df_plot.drop(['Geography_sort'], axis = 1)

df_plot = df_plot[df_plot['Sector'] == 'Government']
df_plot['Percentage'] = round(df_plot['Percentage'], 1)

display(df_plot.head())


## Plotting ---


color_map = {
    'Sacramento-Roseville-Folsom, CA': "#9DC209"
    , 'Yuba City, CA': "#9DC209"
    , 'National': "#1F45FC"
    , 'Austin-Round Rock-Georgetown, TX': "#1E90FF"
    , 'Charlotte-Concord-Gastonia, NC-SC': "#1E90FF"
    , 'Cincinnati, OH-KY-IN': "#1E90FF"
    , 'Cleveland-Elyria, OH': "#1E90FF"
    , 'Columbus, OH': "#1E90FF"
    , 'Detroit-Warren-Dearborn, MI': "#1E90FF"
    , 'Indianapolis-Carmel-Anderson, IN': "#1E90FF"
    , 'Kansas City, MO-KS': "#1E90FF"
    , 'Miami-Fort Lauderdale-Pompano Beach, FL': "#1E90FF"
    , 'Orlando-Kissimmee-Sanford, FL': "#1E90FF"
    , 'Phoenix-Mesa-Chandler, AZ': "#1E90FF"
    , 'Pittsburgh, PA': "#1E90FF"
    , 'Portland-Vancouver-Hillsboro, OR-WA': "#1E90FF"
    , 'Riverside-San Bernardino-Ontario, CA': "#1E90FF"
    , 'Salt Lake City, UT': "#1E90FF"
    , 'San Antonio-New Braunfels, TX': "#1E90FF"
    , 'San Diego-Chula Vista-Carlsbad, CA': "#1E90FF"
    , 'San Francisco-Oakland-Berkeley, CA': "#1E90FF"
    , 'San Jose-Sunnyvale-Santa Clara, CA': "#1E90FF"
    , 'St. Louis, MO-IL': "#1E90FF"
    , 'Tampa-St. Petersburg-Clearwater, FL': "#1E90FF"
}


fig = px.bar(df_plot, y='Geography', x='Percentage'
             , color='Geography'
             , color_discrete_map=color_map
             , orientation='h')

ticktext = []
for geography in df_plot['Geography'].unique():
    if geography in ['Sacramento-Roseville-Folsom, CA', 'Yuba City, CA', 'National']:
        ticktext.append(f'<b>{geography}</b>')
    else:
        ticktext.append(geography)
        
fig.update_layout(yaxis=dict(tickmode='array', tickvals=df_plot['Geography'].unique(), ticktext=ticktext))

title = 'Government Share of Total Regional Jobs, 2024'
fig.update_layout(legend_title=None, title=title, template=template, font_family=font_family)
fig.update_layout(xaxis_title = None, yaxis_title = None, yaxis=dict(tickfont = dict(size=12)), xaxis=dict(tickfont = dict(size=12)), showlegend = False)
fig.update_xaxes(tick0=0, dtick=5, ticksuffix='%')
fig.update_traces(hovertemplate='Government: %{x}')


fig.show()
# fig.write_html(os.path.join(path_plots, 'Jobs_3_public.html'))

In [ ]:
## Importing ---

indicator_name = 'Jobs_3'

file_name1 = f"{indicator_name} MSA BLS SM.xlsx"
file_name2 = f"{indicator_name} National BLS CE.xlsx"

df_msa = pd.read_excel(os.path.join(path_plots, 'Data', file_name1), sheet_name = 'Government and Private')
df_nat = pd.read_excel(os.path.join(path_plots, 'Data', file_name2), sheet_name = 'Government and Private')



## Organizing ---
df_plot = df_msa.copy()

df_plot = df_plot.rename(columns = {'MSA':'Geography'})
df_plot = pd.concat([df_plot, df_nat])

df_plot = df_plot[df_plot['Sector'] != 'All']
df_plot = df_plot[df_msa['date_'] == '2024-04-01']
df_plot['Year'] = pd.to_datetime(df_plot['date_'])
df_plot['Year'] = df_plot['Year'].dt.year
df_plot = df_plot.drop(['date_'], axis = 1)
df_plot = df_plot.set_index('Year').reset_index()
df_plot = df_plot.reset_index(drop = True)

def remove_metro(x):
    x = re.sub(' Metro Area', '', x)
    return x
df_plot['Geography'] = df_plot['Geography'].apply(remove_metro)

df_plot['Geography_sort'] = pd.Categorical(df_plot['Geography'], [
    'Sacramento-Roseville-Folsom, CA'
    , 'Yuba City, CA'
    , 'National'
    , 'Austin-Round Rock-Georgetown, TX'
    , 'Charlotte-Concord-Gastonia, NC-SC'
    , 'Cincinnati, OH-KY-IN'
    , 'Cleveland-Elyria, OH'
    , 'Columbus, OH'
    , 'Detroit-Warren-Dearborn, MI'
    , 'Indianapolis-Carmel-Anderson, IN'
    , 'Kansas City, MO-KS'
    , 'Miami-Fort Lauderdale-Pompano Beach, FL'
    , 'Orlando-Kissimmee-Sanford, FL'
    , 'Phoenix-Mesa-Chandler, AZ'
    , 'Pittsburgh, PA'
    , 'Portland-Vancouver-Hillsboro, OR-WA'
    , 'Riverside-San Bernardino-Ontario, CA'
    , 'Salt Lake City, UT'
    , 'San Antonio-New Braunfels, TX'
    , 'San Diego-Chula Vista-Carlsbad, CA'
    , 'San Francisco-Oakland-Berkeley, CA'
    , 'San Jose-Sunnyvale-Santa Clara, CA'
    , 'St. Louis, MO-IL'
    , 'Tampa-St. Petersburg-Clearwater, FL'
])
    
df_plot = df_plot.sort_values(['Geography_sort', 'Sector'], ascending = [False, True])
df_plot = df_plot.drop(['Geography_sort'], axis = 1)

df_plot['Percentage'] = round(df_plot['Percentage'], 1)

display(df_plot.head())

## Plotting ---

color_map = {
    'Government': '#1E90FF'
    , 'Total Private':'#9DC209'
}

fig = px.bar(df_plot, y='Geography', x='Percentage'
             , color='Sector'
             , color_discrete_map=color_map
             , orientation='h')


title = 'Economic Structure: Share of Public vs Private by Peer Region, 2024'
fig.update_layout(legend_title=None, title=title, template=template, font_family=font_family)
fig.update_layout(xaxis_title = None, yaxis_title = None, yaxis=dict(tickfont = dict(size=12)), xaxis=dict(tickfont = dict(size=12)))
fig.update_xaxes(tick0=0, dtick=10, ticksuffix='%')
fig.update_traces(hovertemplate='%{x}')
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=-0.15,xanchor="right", x=0.6))


fig.show()


***

Labor_2

***

In [ ]:
# Set indicator
indicator_name = 'Labor_2'

file_name = f"{indicator_name} MSA BLS LA.xlsx"
df_msa = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'MSA')
df_jobs = pd.read_excel(os.path.join(path_plots, 'Data', 'Jobs_1 MSA BLS SM.xlsx'), sheet_name = 'MSA')
df_msa = df_msa.rename(columns = {'MSA_ID':'MSA ID'})

display(df_msa.head(), df_jobs.head())

In [ ]:
## Importing ---
indicator_name = 'Labor_2'

file_name = f"{indicator_name} MSA BLS LA.xlsx"
df_msa = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'MSA')
df_jobs = pd.read_excel(os.path.join(path_plots, 'Data', 'Jobs_1 MSA BLS SM.xlsx'), sheet_name = 'MSA')


## Organizing ---

df_msa = df_msa.rename(columns = {'MSA_ID':'MSA ID'})

df_jobs = df_jobs[df_jobs['Sector'] == 'All']
df_jobs = df_jobs[['date_', 'MSA ID', 'Total Jobs']].rename(columns = {'area_code':'MSA_ID'})

df_plot = df_msa.copy()
df_plot = df_plot.merge(df_jobs, on = ['date_', 'MSA ID'], how = 'left')
df_plot = df_plot[~df_plot['Total Jobs'].isna()]


def remove_metro(x):
    x = re.sub(' Metropolitan Statistical Area', '', x)
    return x
df_plot['MSA'] = df_plot['MSA'].apply(remove_metro)

conditions = [
    (  df_plot['MSA'] ==  'Sacramento--Roseville--Arden-Arcade, CA') ,
    (  df_plot['MSA'] ==  'Yuba City, CA'                          ) ,
    ( ~df_plot['MSA'].str.contains('Yuba|Sacramento')              )
]
choices = ['Sacramento--Roseville--Arden-Arcade, CA', 'Yuba City, CA', 'Peer MSA']
df_plot["Group"] = np.select(conditions, choices)

df_plot['Year'] = pd.to_datetime(df_plot['date_'])
df_plot['Year'] = df_plot['Year'].dt.year

wm = lambda x: np.average(x, weights = df_plot.loc[x.index, "Total Jobs"]) # weighted average
df_plot = df_plot.groupby(['Year', 'Group'], as_index = False).agg(unemployment_rate = ('Unemployment Rate', wm))
df_plot['unemployment_rate'] = round(df_plot['unemployment_rate'], 1)
df_plot = df_plot.sort_values(['Year', 'Group'], ascending = [False, True])


df_plot['Sort'] = pd.Categorical(df_plot['Group'], [
    'Sacramento--Roseville--Arden-Arcade, CA'
         , 'Yuba City, CA'
         , "Peer MSA"
])
    
df_plot = df_plot.sort_values(['Sort', 'Year'], ascending = [True, False])
df_plot = df_plot.drop(['Sort'], axis = 1)

display(df_plot.head())



## Plotting ---

color_map = {
         'Sacramento--Roseville--Arden-Arcade, CA':"#1E90FF",
         'Yuba City, CA': "#1F45FC",
         "Peer MSA": "#9DC209"
}

fig = px.line(df_plot, x='Year', y='unemployment_rate', color='Group', markers = True, color_discrete_map=color_map)


title = '<b>Unemployment Rate by MSA, 2022</b>'
fig.update_yaxes(dtick=5, ticksuffix='%', range = [0,22])
fig.update_xaxes(dtick=1)
fig.update_layout(xaxis_title = None, yaxis_title = None, yaxis=dict(tickfont = dict(size=12)), xaxis=dict(tickfont = dict(size=12)))
fig.update_layout(title=title, legend_title=None, template=template, font_family=font_family)
fig.update_traces(hovertemplate="%{y}")
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=-0.15,xanchor="right", x=0.65))


fig.show()
# fig.write_html(os.path.join(path_plots, 'Labor_2_unemployment.html'))